In [2]:
import os
import requests
from datetime import datetime, date
import time
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")
POLYGON_API_KEY   = os.getenv("POLYGON_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, POLYGON_API_KEY]):
    raise EnvironmentError("Faltan variables de entorno")

# ---------------------------
# Conexión PostgreSQL
# ---------------------------
engine = create_engine(
    f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

# ---------------------------
# Configuración
# ---------------------------
SNAPSHOT_DATE  = "2026-04-01"   # ← modificar manualmente

MIN_DTE        = 20
MAX_DTE        = 90
MIN_DELTA      = -0.35
MAX_DELTA      = -0.25
CONTRACT_TYPE  = "put"

# ---------------------------
# 1️⃣ Obtener tickers alcistas y neutrales
# ---------------------------
query_tickers = f"""
    SELECT DISTINCT ticker, direccion
    FROM agente.trade_decision_direction
    WHERE   snapshot_date  = DATE '{SNAPSHOT_DATE}'
"""

df_tickers = pd.read_sql(query_tickers, engine)
tickers_validos = df_tickers[["ticker", "direccion"]].values.tolist()

print(f"Snapshot: {SNAPSHOT_DATE}")
print(f"Procesando {len(tickers_validos)} tickers (alcista + neutral)...")

# ---------------------------
# 2️⃣ Consulta Polygon
# ---------------------------
def buscar_opciones_por_ticker(ticker_raiz):
    hoy      = datetime.now()
    url      = f"https://api.polygon.io/v3/snapshot/options/{ticker_raiz}?limit=250&apiKey={POLYGON_API_KEY}"
    resultados = []
    buscando = True

    while url and buscando:
        try:
            r = requests.get(url)
            if r.status_code != 200:
                print(f"Error Polygon {ticker_raiz}: {r.status_code}")
                break

            data      = r.json()
            contracts = data.get("results", [])

            for c in contracts:
                details = c.get("details", {})
                greeks  = c.get("greeks", {})
                day     = c.get("day", {})

                if details.get("contract_type") != CONTRACT_TYPE:
                    continue

                expiry = details.get("expiration_date")
                if not expiry:
                    continue

                fecha_vto = datetime.strptime(expiry, "%Y-%m-%d")
                dte       = (fecha_vto - hoy).days

                if dte > MAX_DTE:
                    buscando = False
                    break

                delta = greeks.get("delta")
                if delta is None or not (MIN_DELTA <= delta <= MAX_DELTA):
                    continue

                if MIN_DTE <= dte <= MAX_DTE:
                    resultados.append({
                        "ticker":         ticker_raiz,
                        "opcion":         details.get("ticker"),
                        "contract_type":  details.get("contract_type"),
                        "strike":         details.get("strike_price"),
                        "vto":            expiry,
                        "dte":            dte,
                        "delta":          round(delta, 4),
                        "gamma":          greeks.get("gamma"),
                        "theta":          greeks.get("theta"),
                        "vega":           greeks.get("vega"),
                        "iv":             c.get("implied_volatility"),
                        "oi":             c.get("open_interest", 0),
                        "volume":         day.get("volume", 0),
                        "vwap":           day.get("vwap"),
                        "close_price":    day.get("close"),
                        "fecha":          date.today()
                    })

            next_url = data.get("next_url")
            url = f"{next_url}&apiKey={POLYGON_API_KEY}" if next_url else None

        except Exception as e:
            print(f"Excepción {ticker_raiz}: {e}")
            break

    return resultados

# ---------------------------
# 3️⃣ Procesar e insertar
# ---------------------------
total_insertados = 0

for ticker, direccion in tickers_validos:
    print(f"\nProcesando {ticker} ({direccion})...")
    data = buscar_opciones_por_ticker(ticker)

    if data:
        df = pd.DataFrame(data)
        df.to_sql(
            "contratos_raw",
            engine,
            schema="agente_opciones",
            if_exists="append",
            index=False
        )
        print(f"  {len(df)} contratos insertados")
        total_insertados += len(df)
    else:
        print(f"  No se encontraron {CONTRACT_TYPE.upper()}s válidos")

    time.sleep(0.25)

print(f"\nFinalizado. Total insertados: {total_insertados}")
print(f"Snapshot usado: {SNAPSHOT_DATE}")

Snapshot: 2026-04-01
Procesando 208 tickers (alcista + neutral)...

Procesando CARG (neutral_bajista)...
  1 contratos insertados

Procesando NYT (none)...
  1 contratos insertados

Procesando PG (none)...
  13 contratos insertados

Procesando MCRI (alcista)...
  2 contratos insertados

Procesando VMC (none)...
  1 contratos insertados

Procesando LYTS (alcista)...
  1 contratos insertados

Procesando MLI (neutral_bajista)...
  2 contratos insertados

Procesando SN (neutral_bajista)...
  2 contratos insertados

Procesando LAUR (neutral_bajista)...
  No se encontraron PUTs válidos

Procesando COKE (alcista)...
  3 contratos insertados

Procesando PKE (neutral_bajista)...
  No se encontraron PUTs válidos

Procesando INCY (neutral_bajista)...
  3 contratos insertados

Procesando IAS (none)...
  No se encontraron PUTs válidos

Procesando FCN (none)...
  2 contratos insertados

Procesando DCI (alcista)...
  No se encontraron PUTs válidos

Procesando TPC (none)...
  1 contratos insertados

P


Procesando ATGE (none)...
  1 contratos insertados

Procesando REYN (none)...
  1 contratos insertados

Procesando YETI (neutral_bajista)...
  1 contratos insertados

Procesando DOCN (none)...
  21 contratos insertados

Procesando TILE (none)...
  1 contratos insertados

Procesando IOSP (alcista)...
  2 contratos insertados

Procesando RCKY (alcista)...
  No se encontraron PUTs válidos

Procesando BANX (none)...
  1 contratos insertados

Procesando JKHY (neutral_bajista)...
  2 contratos insertados

Procesando ANIP (none)...
  1 contratos insertados

Procesando MTD (neutral_bajista)...
  4 contratos insertados

Procesando CI (none)...
  8 contratos insertados

Procesando VRTX (neutral_bajista)...
  10 contratos insertados

Procesando REVG (neutral_bajista)...
  No se encontraron PUTs válidos

Procesando FSS (none)...
  1 contratos insertados

Procesando ALGN (neutral_bajista)...
  9 contratos insertados

Procesando APXT (neutral_bajista)...
  No se encontraron PUTs válidos

Procesando